# Notebook 02b: BindCraft → ProteinMPNN → Protenix → Contact Filter

```
[NB01] rank0X_filtered_YYYY.pdb  ← upload one conformer per session
    ↓
BindCraft  (AF2 multimer internally)
  → complex PDB + binder sequence + ipTM
    ↓
ProteinMPNN on complex
  → 4–8 redesigned sequences per backbone
  (target chain fixed, binder chain redesigned)
    ↓
Protenix  (AF3-level complex prediction, no MSA)
  → CIF structure + iptm + ranking_score
  → epitope contact analysis from CIF
    ↓
bindcraft_rank0X_passing.csv + CIF files  ← feed into NB03
```

**Run once per conformer.** Requires GPU (T4).  
Runtime: ~3–5h per conformer (BindCraft dominates).

## 0. GPU check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch to GPU runtime')

## 1. Install

In [ ]:
%%capture
import os
# BindCraft (AF2 + PyRosetta)
!git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
os.chdir('/content/BindCraft')
!bash install_bindcraft.sh

# ProteinMPNN
!git clone https://github.com/dauparas/ProteinMPNN /content/ProteinMPNN

# Protenix (AF3-level complex prediction)
!pip install protenix

# Utilities
!pip install biopython pandas matplotlib

In [ ]:
import importlib, os, subprocess
for pkg in ['pyrosetta', 'biopython']:
    try:
        importlib.import_module(pkg)
        print(f'{pkg}: OK')
    except ImportError:
        print(f'{pkg}: MISSING')
print('ProteinMPNN:', 'OK' if os.path.exists('/content/ProteinMPNN/protein_mpnn_run.py') else 'MISSING')
r = subprocess.run(['protenix', '--help'], capture_output=True, text=True)
print('Protenix:', 'OK' if r.returncode == 0 else 'MISSING')

## 2. Configuration

In [ ]:
# ── BindCraft ─────────────────────────────────────────────────────────────────
N_BINDCRAFT_DESIGNS = 50
BINDER_LEN_MIN      = 50
BINDER_LEN_MAX      = 80
TARGET_CHAIN        = 'A'
BINDER_CHAIN        = 'B'

EPITOPE_RANGES = [
    (2,  7,  'PHF6* VQIINK', True),
    (33, 38, 'PHF6 VQIVYK',  True),
    (52, 70, 'jR2R3',        False),
]

# ── ProteinMPNN ───────────────────────────────────────────────────────────────
MPNN_SEQS_PER_STRUCT = 8
MPNN_SAMPLING_TEMP   = 0.1

# ── Protenix ──────────────────────────────────────────────────────────────────
# protenix_mini_default_v0.5.0 → 134M params, no MSA, fits T4 at ~250 tokens
# protenix_base_default_v1.0.0 → 368M params, higher accuracy, needs ~18GB for 1k tokens
PROTENIX_MODEL     = 'protenix_mini_default_v0.5.0'
PROTENIX_SEEDS     = '101'       # comma-separated seeds, 1 = fast

# K18 target sequence (used as second chain in Protenix complex prediction)
K18_SEQUENCE = (
    'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGG'
    'QVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPV'
    'VSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL'
)

# ── Filter thresholds ─────────────────────────────────────────────────────────
# ranking_score = 0.8*iptm + 0.2*ptm (AF3 equivalent of AF2-multimer ranking)
MIN_RANKING_SCORE    = 0.5
MIN_EPITOPE_CONTACTS = 1
CONTACT_DIST_A       = 8.0

# ─────────────────────────────────────────────────────────────────────────────
import os, json
for d in ['output/bindcraft','output/mpnn','output/protenix','output/passing']:
    os.makedirs(f'/content/{d}', exist_ok=True)

hotspot_res = ','.join(
    f'{TARGET_CHAIN}{r}'
    for start, end, label, include in EPITOPE_RANGES if include
    for r in range(start, end+1)
)

epitope_res_map = {}
for start, end, label, *_ in EPITOPE_RANGES:
    for r in range(start, end+1):
        epitope_res_map[r] = label

print(f'Hotspot residues : {hotspot_res}')
print(f'Protenix model   : {PROTENIX_MODEL}')
print(f'K18 length       : {len(K18_SEQUENCE)} aa')

## 3. Upload target conformer

In [ ]:
try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    print('Upload one rank0X_filtered_YYYY.pdb from NB01 zip...')
    uploaded = _colab_files.upload()
    pdb_files = [f for f in uploaded if f.endswith('.pdb')]
    if not pdb_files:
        raise ValueError('No PDB uploaded.')
    TARGET_PDB_PATH = f'/content/{pdb_files[0]}'
    with open(TARGET_PDB_PATH, 'wb') as f:
        f.write(uploaded[pdb_files[0]])
    CONFORMER_NAME = os.path.splitext(pdb_files[0])[0]
else:
    # Local fallback — set TARGET_PDB_PATH manually
    TARGET_PDB_PATH = 'results/nb01_production/tau_k18_binder_ready/rank01_filtered_0047.pdb'
    CONFORMER_NAME = os.path.splitext(os.path.basename(TARGET_PDB_PATH))[0]

print(f'Conformer: {CONFORMER_NAME}')
print(f'PDB path : {TARGET_PDB_PATH}')


## 4. Run BindCraft (~2–4h)

In [ ]:
import time, pandas as pd, numpy as np, glob

bc_out = f'/content/output/bindcraft/{CONFORMER_NAME}'
os.makedirs(bc_out, exist_ok=True)

settings = {
    'design_path': bc_out, 'binder_len': f'{BINDER_LEN_MIN}-{BINDER_LEN_MAX}',
    'hotspot_res': hotspot_res, 'num_designs': N_BINDCRAFT_DESIGNS,
    'use_multimer': True, 'refine_designs': True,
    'target_pdb': TARGET_PDB_PATH, 'target_chain': TARGET_CHAIN,
}
settings_path = f'/content/output/bindcraft/settings_{CONFORMER_NAME}.json'
with open(settings_path, 'w') as f:
    json.dump(settings, f, indent=2)

print('Running BindCraft...')
t0 = time.time()
# BUG-17 fix: check returncode so BindCraft failures surface immediately
result = subprocess.run(
    ['python', '/content/BindCraft/bindcraft.py', '--settings', settings_path],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("=== BindCraft STDERR (last 3000 chars) ===")
    print(result.stderr[-3000:])
    raise RuntimeError(f"BindCraft exited with code {result.returncode}. See stderr above.")
print(f'BindCraft done in {(time.time()-t0)/60:.1f} min')

# Load and standardize output CSV
csv_candidates = glob.glob(f'{bc_out}/**/*.csv', recursive=True)
if not csv_candidates:
    raise FileNotFoundError(f'No BindCraft CSV found in {bc_out} — BindCraft may have failed silently.')
bc = pd.read_csv(csv_candidates[0])
print(f'BindCraft designs: {len(bc)}  columns: {list(bc.columns[:6])}')


## 5. ProteinMPNN — redesign sequences on BindCraft complex backbones

In [ ]:
mpnn_out = f'/content/output/mpnn/{CONFORMER_NAME}'
os.makedirs(mpnn_out, exist_ok=True)

def run_mpnn_on_complex(pdb_path, out_dir, n_seqs=MPNN_SEQS_PER_STRUCT,
                        temp=MPNN_SAMPLING_TEMP):
    stem    = os.path.splitext(os.path.basename(pdb_path))[0]
    run_dir = os.path.join(out_dir, stem)
    os.makedirs(run_dir, exist_ok=True)

    jsonl = os.path.join(run_dir, 'input.jsonl')
    with open(jsonl, 'w') as f:
        f.write(json.dumps({
            'PDB': pdb_path,
            'fixed_chains': [TARGET_CHAIN],
            'designed_chains': [BINDER_CHAIN]
        }) + '\n')

    subprocess.run([
        'python', '/content/ProteinMPNN/protein_mpnn_run.py',
        '--jsonl_path', jsonl, '--out_folder', run_dir,
        '--num_seq_per_target', str(n_seqs),
        '--sampling_temp', str(temp), '--batch_size', '1',
    ], capture_output=True, text=True)

    seqs = []
    for fa in glob.glob(os.path.join(run_dir, 'seqs', '*.fa')):
        lines = open(fa).readlines()
        for i, line in enumerate(lines):
            if line.startswith('>') and i+1 < len(lines):
                parts = lines[i+1].strip().split('/')
                # BUG-18 fix: validate by length rather than assuming chain order
                binder_candidates = [p for p in parts
                                     if BINDER_LEN_MIN <= len(p) <= BINDER_LEN_MAX]
                if not binder_candidates:
                    print(f"  WARNING: No MPNN part in binder length range "
                          f"[{BINDER_LEN_MIN},{BINDER_LEN_MAX}]. "
                          f"Part lengths: {[len(p) for p in parts]}. Skipping.")
                    continue
                seqs.append(binder_candidates[0])
    return seqs

print(f'ProteinMPNN on {len(pdb_lookup)} BindCraft complexes...')
mpnn_records = []
for bc_name, pdb_path in pdb_lookup.items():
    seqs   = run_mpnn_on_complex(pdb_path, mpnn_out)
    bc_row = bc[bc['name']==bc_name]
    iptm   = float(bc_row['iptm'].iloc[0]) if len(bc_row) else 0.5
    for i, seq in enumerate(seqs):
        mpnn_records.append({
            'name': f'{bc_name}_mpnn_{i+1:02d}', 'sequence': seq,
            'source_bc_name': bc_name, 'source_complex_pdb': pdb_path,
            'bc_iptm': iptm, 'method': 'BindCraft',
            'target_conformer': CONFORMER_NAME,
        })

mpnn_df = pd.DataFrame(mpnn_records)
print(f'ProteinMPNN sequences: {len(mpnn_df)}')


## 6. Protenix — predict binder+target complex

Runs AF3-level complex prediction on each ProteinMPNN sequence paired with K18.
No MSA needed — raw sequences only (`--use_msa false`).
Outputs CIF structure + `iptm` + `ranking_score` per binder.

In [ ]:
protenix_out = f'/content/output/protenix/{CONFORMER_NAME}'
os.makedirs(protenix_out, exist_ok=True)

def run_protenix(name, binder_seq, target_seq, out_dir,
                 model=PROTENIX_MODEL, seeds=PROTENIX_SEEDS):
    """
    Predict binder+target complex with Protenix.
    Target is chain A (first), binder is chain B (second).
    Returns (iptm, ranking_score, cif_path) or (None, None, None) on failure.
    """
    run_dir    = os.path.join(out_dir, name)
    os.makedirs(run_dir, exist_ok=True)

    input_data = [{
        'name': name,
        'sequences': [
            {'proteinChain': {'sequence': target_seq,  'count': 1}},  # chain A = target
            {'proteinChain': {'sequence': binder_seq, 'count': 1}},   # chain B = binder
        ]
    }]
    input_json = os.path.join(run_dir, 'input.json')
    with open(input_json, 'w') as f:
        json.dump(input_data, f)

    result = subprocess.run([
        'protenix', 'pred',
        '-i', input_json,
        '-o', run_dir,
        '-n', model,
        '-s', seeds,
        '-e', '1',
        '--use_msa', 'false',
    ], capture_output=True, text=True)

    if result.returncode != 0:
        return None, None, None

    conf_files = glob.glob(f'{run_dir}/**/*summary_confidence*.json', recursive=True)
    if not conf_files:
        return None, None, None

    with open(conf_files[0]) as f:
        conf = json.load(f)

    cif_files = glob.glob(f'{run_dir}/**/*.cif', recursive=True)

    iptm = conf.get('iptm', 0.0)
    ptm  = conf.get('ptm',  0.0)
    # BUG-19 fix: compute ranking_score manually — Protenix may not emit this key directly
    ranking_score = conf.get('ranking_score', 0.8 * iptm + 0.2 * ptm)

    return (iptm, ranking_score, cif_files[0] if cif_files else None)


print(f'Running Protenix on {len(mpnn_df)} sequences...')
print(f'Model: {PROTENIX_MODEL}  |  MSA: disabled')

iptm_vals, ranking_vals, cif_paths = [], [], []
for i, row in mpnn_df.iterrows():
    iptm, ranking, cif = run_protenix(
        row['name'], row['sequence'], K18_SEQUENCE, protenix_out
    )
    iptm_vals.append(iptm or 0.0)
    ranking_vals.append(ranking or 0.0)
    cif_paths.append(cif)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(mpnn_df)}  last ranking_score={ranking:.3f}' if ranking else f'  {i+1}/{len(mpnn_df)}')

mpnn_df['iptm']          = iptm_vals
mpnn_df['ranking_score'] = ranking_vals
mpnn_df['cif_path']      = cif_paths

print(f'\nProtenix done.')
print(f'ranking_score  mean={mpnn_df["ranking_score"].mean():.3f}  std={mpnn_df["ranking_score"].std():.3f}')
print(f'iptm           mean={mpnn_df["iptm"].mean():.3f}')


## 7. Epitope contact analysis from Protenix CIF

Parses the Protenix complex CIF to check which K18 epitope residues
(PHF6* / PHF6 / jR2R3) are contacted by the binder.

In [ ]:
from Bio.PDB import MMCIFParser
from collections import defaultdict
import warnings; warnings.filterwarnings('ignore')
_cif_parser = MMCIFParser(QUIET=True)

def epitope_contacts_from_cif(cif_path, target_chain='A', binder_chain='B',
                               dist_thresh=CONTACT_DIST_A):
    """
    Parse Protenix CIF, return (n_epitope_contacts, breakdown_str, interface_plddt).
    Chain A = target (K18), Chain B = binder.
    """
    if not cif_path or not os.path.exists(cif_path):
        return 0, 'no CIF', 0.0
    try:
        struct = _cif_parser.get_structure('s', cif_path)
        model  = struct[0]
        chain_ids = [c.id for c in model]

        if target_chain not in chain_ids or binder_chain not in chain_ids:
            return 0, f'chains not found: {chain_ids}', 0.0

        t_ca, b_ca = {}, {}
        for res in model[target_chain].get_residues():
            if res.get_id()[0]==' ' and 'CA' in res:
                ca = res['CA']
                t_ca[res.get_id()[1]] = ca.get_vector().get_array()
        for res in model[binder_chain].get_residues():
            if res.get_id()[0]==' ' and 'CA' in res:
                ca = res['CA']
                b_ca[res.get_id()[1]] = (ca.get_vector().get_array(), ca.get_bfactor())

        epi_hits = defaultdict(set)
        contact_b = set()
        for b_res,(b_coord,_) in b_ca.items():
            for t_res, t_coord in t_ca.items():
                if np.linalg.norm(b_coord - t_coord) <= dist_thresh:
                    contact_b.add(b_res)
                    lbl = epitope_res_map.get(t_res)
                    if lbl:
                        epi_hits[lbl].add(t_res)

        iface_plddt = np.mean([b_ca[r][1] for r in contact_b]) if contact_b else 0.0
        total_epi   = sum(len(v) for v in epi_hits.values())
        breakdown   = ' | '.join(f'{l}: {len(r)}' for l,r in epi_hits.items())
        return total_epi, breakdown, round(float(iface_plddt), 1)
    except Exception as e:
        return 0, str(e), 0.0


epi_contacts, epi_breakdowns, iface_plddts = [], [], []
for _, row in mpnn_df.iterrows():
    n, bd, ip = epitope_contacts_from_cif(row['cif_path'])
    epi_contacts.append(n)
    epi_breakdowns.append(bd)
    iface_plddts.append(ip)

mpnn_df['epitope_contacts']   = epi_contacts
mpnn_df['epitope_breakdown']  = epi_breakdowns
mpnn_df['interface_plddt']    = iface_plddts

# Apply filters
mpnn_df['verdict'] = 'DISCARD'
mpnn_df.loc[
    (mpnn_df['ranking_score'] >= MIN_RANKING_SCORE) &
    (mpnn_df['epitope_contacts'] >= MIN_EPITOPE_CONTACTS),
    'verdict'
] = 'PASS'

n_pass = (mpnn_df['verdict']=='PASS').sum()
print(f'PASS: {n_pass}/{len(mpnn_df)} ({100*n_pass/len(mpnn_df):.0f}%)')
print(f'  ranking_score ≥ {MIN_RANKING_SCORE}  AND  epitope_contacts ≥ {MIN_EPITOPE_CONTACTS}')
print()
print(mpnn_df[mpnn_df['verdict']=='PASS'][
    ['name','iptm','ranking_score','epitope_contacts','epitope_breakdown']
].head(10).to_string(index=False))

## 8. QC plots + save + download

In [ ]:
import matplotlib.pyplot as plt, shutil, zipfile

pass_df = mpnn_df[mpnn_df['verdict']=='PASS'].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
ax.hist(mpnn_df['ranking_score'], bins=20, color='steelblue', alpha=0.7, label='All')
ax.hist(pass_df['ranking_score'], bins=20, color='seagreen',  alpha=0.7, label='Passing')
ax.axvline(MIN_RANKING_SCORE, color='tomato', ls='--', lw=2, label=f'threshold={MIN_RANKING_SCORE}')
ax.set_xlabel('Protenix ranking_score'); ax.set_title('Ranking score'); ax.legend(fontsize=8)
ax = axes[1]
ax.scatter(mpnn_df['iptm'], mpnn_df['ranking_score'],
           c=['seagreen' if v=='PASS' else 'tomato' for v in mpnn_df['verdict']],
           s=30, alpha=0.7)
ax.set_xlabel('iptm'); ax.set_ylabel('ranking_score'); ax.set_title('iptm vs ranking_score')
ax = axes[2]
ax.scatter(pass_df['epitope_contacts'], pass_df['ranking_score'],
           c=pass_df['interface_plddt'], cmap='RdYlGn', s=50)
ax.set_xlabel('Epitope contacts'); ax.set_ylabel('ranking_score')
ax.set_title('Passing: contacts vs score\n(color=interface pLDDT)')
plt.suptitle(f'BindCraft pipeline — {CONFORMER_NAME}', fontsize=12)
plt.tight_layout()
qc_png = f'/content/bindcraft_qc_{CONFORMER_NAME}.png'
plt.savefig(qc_png, dpi=150, bbox_inches='tight')
plt.show()

# Save CSV
out_cols = ['name','sequence','iptm','ranking_score','plddt','method','target_conformer',
            'epitope_contacts','epitope_breakdown','interface_plddt','verdict']
out_cols = [c for c in out_cols if c in pass_df.columns]
passing_csv = f'/content/bindcraft_{CONFORMER_NAME}_passing.csv'
pass_df[out_cols].to_csv(passing_csv, index=False)

# Copy CIF files for passing binders
pass_dir = '/content/output/passing'
for _, row in pass_df.iterrows():
    if row.get('cif_path') and os.path.exists(row['cif_path']):
        shutil.copy2(row['cif_path'], f"{pass_dir}/{row['name']}.cif")

# Zip
zip_name = f'/content/bindcraft_{CONFORMER_NAME}.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(passing_csv, os.path.basename(passing_csv))
    zf.write(qc_png, os.path.basename(qc_png))
    for fn in os.listdir(pass_dir):
        zf.write(f'{pass_dir}/{fn}', f'cifs/{fn}')

print(f'Passing: {len(pass_df)} binders → {passing_csv}')
print(f'Repeat for rank02–rank05, then upload all CSVs to NB03.')

try:
    from google.colab import files as _colab_files
    _colab_files.download(zip_name)
except ImportError:
    print(f'Not in Colab — file saved at: {zip_name}')
